# Production RAG Prototype — College Chatbot (Google Colab)

Colab version of the Phase 1 prototype (see `README.md` at the repo root).

**Pipeline:** load docs → chunk → embed → store in Pinecone → retrieve top-k → generate a cited answer.

**Stack:** LangChain, OpenAI (embeddings + LLM), Pinecone (vector store).

**Before running:** add `OPENAI_API_KEY` and `PINECONE_API_KEY` as Colab secrets
(the 🔑 key icon in the left sidebar) and toggle "Notebook access" on for both. This keeps your
keys out of the notebook file itself — never hardcode them in a cell.

In [ ]:
!pip install -qU langchain langchain-openai langchain-pinecone langchain-community pinecone python-dotenv pypdf tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

## 1. Credentials

Reads from Colab Secrets first (recommended). Falls back to a masked prompt if a secret isn't
found or you're not running in Colab (e.g. running this same notebook locally).

In [ ]:
import os

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def get_secret(name):
    if IN_COLAB:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    # Fallback: masked manual entry
    import getpass
    return getpass.getpass(f"Enter {name}: ")

OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
PINECONE_API_KEY = get_secret("PINECONE_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

print("Keys loaded:", bool(OPENAI_API_KEY), bool(PINECONE_API_KEY))

Enter OPENAI_API_KEY: ··········
Enter PINECONE_API_KEY: ··········
Keys loaded: True True


## 2. Load documents

Two options — pick whichever fits your workflow:

- **Option A (quick):** upload files directly from your computer for this session.
- **Option B (persistent):** mount Google Drive and point at a folder there, so you don't
  re-upload every session.

Option A is set up below. To switch to Drive, comment it out and uncomment Option B.

In [ ]:
import os

os.makedirs("data", exist_ok=True)

# ---- Option A: direct upload (files live only for this Colab session) ----
from google.colab import files

print("Select the PDF(s)/Markdown file(s) to upload (e.g. ctu_training_solutions_overview.pdf):")
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, os.path.join("data", filename))
print("Files in ./data:", os.listdir("data"))

# ---- Option B: Google Drive (uncomment to use instead) ----
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/AI Engineer Portfolio Projects/1-production-rag/data"
# os.makedirs(DATA_DIR, exist_ok=True)

Select the PDF(s)/Markdown file(s) to upload (e.g. ctu_training_solutions_overview.pdf):


Saving ctu_training_solutions_overview.pdf to ctu_training_solutions_overview.pdf
Files in ./data: ['ctu_training_solutions_overview.pdf']


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader

DATA_DIR = "data"  # change to DATA_DIR from Option B above if using Drive

pdf_loader = DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyPDFLoader)
pdf_docs = pdf_loader.load()

md_loader = DirectoryLoader(DATA_DIR, glob="**/*.md", loader_cls=TextLoader)
md_docs = md_loader.load()

raw_docs = pdf_docs + md_docs
print(f"Loaded {len(raw_docs)} raw documents")

/tmp/ipykernel_4881/2869415078.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader


Loaded 8 raw documents


## 3. Chunk

500–800 tokens per chunk, ~100 token overlap, so we don't slice a sentence in half at a chunk
boundary. Using a token-aware splitter (tiktoken) rather than raw character count.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=700,      # tokens
    chunk_overlap=100,   # tokens
)

chunks = text_splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")
chunks[0] if chunks else None

Split into 8 chunks


Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-10T12:46:44+02:00', 'author': 'Compiled for CTU Chatbot RAG corpus', 'keywords': '', 'moddate': '2026-08-10T12:46:44+02:00', 'subject': '(unspecified)', 'title': 'CTU Training Solutions - Institutional Overview', 'trapped': '/False', 'source': 'data/ctu_training_solutions_overview.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="CTU Training Solutions\nInstitutional Overview & Reference Document\nCompiled from CTU Training Solutions' official website (ctutraining.ac.za) and publicly available sources, August 2026.\nIntended as a reference corpus for a retrieval-augmented generation (RAG) chatbot. Prospective students and staff\nshould always confirm current fees, dates, and requirements directly with CTU, as these change year to year.")

## 4. Embeddings + Pinecone index

Creates the Pinecone index if it doesn't exist yet (serverless, cosine similarity, 1536 dims for
`text-embedding-3-small`).

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings

pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "college-rag-chatbot"

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 5. Store chunks in Pinecone

In [ ]:
from uuid import uuid4
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

# Only run this once per fresh batch of docs -- re-running will re-embed and duplicate ids
# unless you pass deterministic ids.
ids = [str(uuid4()) for _ in range(len(chunks))]
vector_store.add_documents(documents=chunks, ids=ids)
print(f"Upserted {len(chunks)} chunks into '{INDEX_NAME}'")

Upserted 8 chunks into 'college-rag-chatbot'


## 6. Retrieval + cited generation

Top-k retrieval, then a prompt that forces the model to ground its answer in the retrieved chunks
and cite the source. This is the Phase 1 bar: show a question, show the answer, point to the
exact source paragraph.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant answering questions about the college using ONLY the
context below. Every claim in your answer must be traceable to the context.

- If the context does not contain enough information to answer, say so explicitly instead
  of guessing.
- After your answer, list the sources you used with their metadata (e.g. filename/page).

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown')} | page {d.metadata.get('page', 'n/a')}]\n{d.page_content}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

In [ ]:
question = "What is CTU?"
answer = rag_chain.invoke(question)
print(answer)

CTU Training Solutions is a private higher education institution in South Africa that has been developing skilled professionals since 1987. It is registered with the Department of Higher Education and Training (DHET) and accredited by the Council on Higher Education (CHE). CTU is a member of the UXi Private Education Group and aims to empower people through education by providing an exceptional learning experience that contributes to socio-economic growth.

Sources:
- [data/ctu_training_solutions_overview.pdf | page 1.0]


## 7. Sanity check retrieval directly

Useful for debugging when an answer looks wrong — see exactly which chunks were retrieved
before the LLM ever saw them.

In [ ]:
retrieved = retriever.invoke(question)
for i, d in enumerate(retrieved, 1):
    print(f"--- Chunk {i} | {d.metadata.get('source')} ---")
    print(d.page_content[:300], "...\n")

--- Chunk 1 | data/ctu_training_solutions_overview.pdf ---
free internet access, printing, and photocopying.
5. Admissions & Application Process
How to Apply

Visit ctutraining.ac.za and browse available study methods (full-time, part-time, online).

Complete the free online application form and upload required documents.

Submit the application — an adm ...

--- Chunk 2 | data/ctu_training_solutions_overview.pdf ---

Last date of enrolment for legacy qualifications: 30 June 2024.

Last date of achievement for legacy qualifications: extended to 30 June 2027.

Achievement of a legacy qualification remains valid and is recorded on the SAQA National Learner
Database.

Students currently on legacy qualifications ...

--- Chunk 3 | data/ctu_training_solutions_overview.pdf ---
Assessment: The process of gathering and evaluating evidence of a student's performance against agreed
criteria to judge whether required learning outcomes have been achieved. Students are assessed both
formativ

## Next steps (Phase 2, per README)

- Add BM25 keyword search and combine with this vector retriever (hybrid retrieval).
- Add a cross-encoder reranker on top of the combined candidates.
- Add citation enforcement: detect when retrieved chunks don't actually support the answer and
  have the model decline instead of guessing.
- Move the prompt above into a versioned config file.